# Battery Digital Twin — PINN Training (Revised v2)

This notebook rebuilds the original Databricks PINN battery digital twin pipeline, with five fixes applied:

1. **Cell 11 (feature engineering):** logs how many cycles are skipped and why, instead of silently dropping them.
2. **OCV model:** replaced the crude linear `U_ocv = 2.7 + 0.015 * SoC` with a fitted empirical curve that captures the nonlinear knees near 0% and 100% SoC.
3. **`log_A_sei` persistence:** the learned Arrhenius pre-exponential factor is now saved *with* the network checkpoint and restored at inference time, instead of resetting to its untrained default.
4. **Inference-time architecture reconstruction:** reuses the real physics PDE (instead of a `dummy_pde` placeholder) when rebuilding the network for weight loading, with an explicit note on what `model.predict()` does and does not enforce.
5. **GPU-native interpolation:** `get_real_time_physics` no longer round-trips through `scipy.interp1d` on the CPU for every PDE residual call — it now uses a `torch.searchsorted`-based linear interpolation that stays entirely on-device, removing a CPU/GPU sync bottleneck that likely dominated training time more than GPU tier choice.

Sections:
- Setup
- Data Engineering (Bronze → Silver → Gold, Spark SQL)
- Feature Engineering (SoC / SoH / cycle downsampling)
- Physics Definition (PDE system, OCV fit, constants)
- Geometry & Boundary/Initial Conditions
- Model Architecture & Training
- Digital Twin Restoration & Abuse-Test Inference

**Note on `%pip install`:** `scipy` is still installed for the initial `polyfit`/setup convenience,
but is no longer required inside the training loop — the interpolation used during PDE residual
evaluation is now pure PyTorch.


## 1. Setup

In [0]:
# Installing PyTorch and DeepXDE
%pip install torch torchvision torchaudio
%pip install deepxde
%pip install scipy

# This restarts the Python process so it recognizes the new libraries
dbutils.library.restartPython()

In [0]:
import os
# Set the backend to pytorch
os.environ["DDE_BACKEND"] = "pytorch"

import torch
import deepxde as dde

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm import tqdm
import mlflow

# Verify installation
print(f"DeepXDE backend: {dde.backend.backend_name}")
print(f"PyTorch version: {torch.__version__}")

## 2. Data Engineering (Medallion Architecture)

Bronze -> Silver -> Gold, built with Spark SQL exactly as before. Battery IDs `B0049`–`B0052`
are excluded (kept consistent with the original notebook's exclusion list).

In [0]:
# Load the raw files into Spark DataFrames
raw_metadata = spark.read.csv("/Volumes/workspace/default/pinn/cleaned_battery_dataset/metadata.csv", header=True, inferSchema=True)

# Register as Temporary Views so we can use SQL
raw_metadata.createOrReplaceTempView("raw_metadata")

# Save as a permanent Bronze Table
raw_metadata.write.format("delta").mode("overwrite").saveAsTable("bronze_battery_metadata")

In [0]:
%sql
SELECT *
FROM raw_metadata
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_discharge_metadata AS
WITH exploded_time AS (
  SELECT
    battery_id,
    type,
    filename,
    ambient_temperature,
    start_time,
    capacity,
    regexp_extract_all(start_time, r'(\d+\.?\d*([eE][+-]?\d+)?)') as parts
  FROM bronze_battery_metadata
  WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
),
formatted_time AS (
  SELECT *,
    CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
    CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
    CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
    CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
    CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
    CAST(parts[5] AS DOUBLE) as sec
  FROM exploded_time
)
SELECT
  battery_id, type, filename, ambient_temperature, capacity,
  to_timestamp(concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec), 'yyyy-M-d H:m:s.SSS') AS start_datetime,
  ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
FROM formatted_time
WHERE type = 'discharge';

In [0]:
%sql
SELECT *
FROM silver_discharge_metadata
WHERE start_datetime IS NOT NULL
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_impedance_metadata AS
WITH cleaned_strings AS (
  SELECT
    battery_id,
    Re,
    Rct,
    type,
    -- Strip brackets and split by one or more spaces
    split(trim(regexp_replace(start_time, r'\[|\]', '')), ' +') as parts
  FROM raw_metadata
  WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
),
formatted_time AS (
  SELECT
    *,
    -- Convert indices to numbers, handling potential scientific notation
    CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
    CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
    CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
    CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
    CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
    CAST(parts[5] AS DOUBLE) as sec
  FROM cleaned_strings
  -- Ensure we actually have 6 parts (Y, M, D, H, M, S)
  WHERE size(parts) >= 6
)
SELECT
  battery_id,
  Re,
  Rct,
  to_timestamp(
    concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec),
    'yyyy-M-d H:m:s.SSS'
  ) AS start_datetime,
  ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
FROM formatted_time
WHERE type = 'impedance';

In [0]:
%sql
SELECT *
FROM silver_impedance_metadata
WHERE start_datetime IS NOT NULL
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_battery_pinn_database AS
SELECT
  d.battery_id,
  d.type,
  d.filename,
  d.ambient_temperature,
  d.start_datetime,
  d.cycle_number,
  CAST(d.capacity AS FLOAT) AS capacity,
  CAST(i.Re AS FLOAT) AS Re,
  CAST(i.Rct AS FLOAT) AS Rct
FROM silver_discharge_metadata d
LEFT JOIN silver_impedance_metadata i
  ON d.battery_id = i.battery_id
  AND i.start_datetime <= d.start_datetime
-- Keep only the first occurrence of each filename
-- Sorting by start_datetime ensures we keep the most logical chronological record
QUALIFY ROW_NUMBER() OVER(PARTITION BY d.filename ORDER BY d.start_datetime ASC) = 1;

In [0]:
%sql
SELECT *
FROM gold_battery_pinn_database
WHERE start_datetime IS NOT NULL
ORDER BY filename
LIMIT 50;

In [0]:
%sql
SELECT `battery_id`, COUNT(*) AS `count`
FROM `workspace`.`default`.`gold_battery_pinn_database`
WHERE `ambient_temperature` = 24
GROUP BY `battery_id`
ORDER BY `count` DESC;

## 3. Feature Engineering (SoC / SoH / Cycle Downsampling)

**Fix applied:** the original silently skipped cycles below the `1.4 Ah` capacity floor and any
cycle shorter than 50 samples, with no visibility into how much data was dropped. This version
counts and prints every skip reason, plus the final retained fraction, so silent data loss can't
go unnoticed.

In [0]:
# --- Configurations ---
cutoff_map = {
    'B0005': 2.7,'B0006': 2.5,'B0007': 2.2,'B0018': 2.5,'B0025': 2.0, 'B0026': 2.2,
    'B0027': 2.5,'B0028': 2.7,'B0029': 2.0,'B0030': 2.2,'B0031': 2.5,'B0032': 2.7,
    'B0033': 2.0,'B0034': 2.2,'B0036': 2.7,'B0038': 2.2,'B0039': 2.5,'B0040': 2.7,
    'B0041': 2.0,'B0042': 2.2,'B0043': 2.5,'B0044': 2.7,'B0045': 2.0,'B0046': 2.2,
    'B0047': 2.5,'B0048': 2.7,'B0053': 2.0,'B0054': 2.2,'B0055': 2.5,'B0056': 2.7
}

target_batteries = ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027',
                    'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034',
                    'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043',
                    'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054',
                    'B0055', 'B0056']

# Minimum capacity (Ah) for a cycle to be considered "healthy enough" to trust.
# Named as a constant rather than a bare literal so its purpose is documented
# and it's easy to tune per-chemistry.
MIN_VALID_CAPACITY_AH = 1.4
MIN_SAMPLES_PER_CYCLE = 50

# --- Query the Gold Database ---
query = f"""
    SELECT * FROM gold_battery_pinn_database
    WHERE battery_id IN ({','.join([f"'{b}'" for b in target_batteries])})
"""
discharge_metadata = spark.sql(query).toPandas()

# Run once outside the loop
max_caps_df = spark.sql("""
    SELECT battery_id, max(capacity) as max_cap
    FROM `workspace`.`default`.`gold_battery_pinn_database`
    GROUP BY battery_id
""").toPandas().set_index('battery_id')

processed_dfs = []

# --- Skip-reason counters (fix: visibility into data loss) ---
skip_counts = {
    'file_not_found': 0,
    'too_short': 0,
    'low_capacity': 0,
}

# --- Process Discharge Cycles ---
for _, row in tqdm(discharge_metadata.iterrows(), total=len(discharge_metadata)):
    file_path = f"/Volumes/workspace/default/pinn/cleaned_battery_dataset/data/{row['filename']}"
    if not os.path.exists(file_path):
        skip_counts['file_not_found'] += 1
        continue

    # Load raw time-series data
    df = pd.read_csv(file_path).copy()

    # Truncate at voltage limit
    v_limit = cutoff_map.get(row['battery_id'], 2.7)
    cutoff_idx = df[df['Voltage_measured'] < v_limit].index.min()
    truncated_df = df if pd.isna(cutoff_idx) else df.iloc[:cutoff_idx].copy()

    if len(truncated_df) < MIN_SAMPLES_PER_CYCLE:
        skip_counts['too_short'] += 1
        continue

    # Calculate Capacity & SoC
    db_capacity = float(row['capacity'])
    max_capacity = max_caps_df.loc[row['battery_id'], 'max_cap']

    # Only process cycles with valid capacity
    if db_capacity <= MIN_VALID_CAPACITY_AH:
        skip_counts['low_capacity'] += 1
        continue

    truncated_df['Time_diff_hr'] = truncated_df['Time'].diff().fillna(0) / 3600
    truncated_df['Delta_Q'] = abs(truncated_df['Current_measured'] * truncated_df['Time_diff_hr'])
    truncated_df['Cumulative_Q'] = truncated_df['Delta_Q'].cumsum()
    truncated_df['SoC'] = 100 * (1 - truncated_df['Cumulative_Q'] / db_capacity)

    # Downsample to 50 points
    num_bins = 50
    truncated_df['bin'] = pd.cut(truncated_df.index, bins=num_bins, labels=False)

    agg_cycle = truncated_df.groupby('bin').agg({
        'Time': 'mean',
        'Voltage_measured': 'mean',
        'Current_measured': 'mean',
        'Temperature_measured': 'mean',
        'SoC': 'mean'
    }).reset_index()

    # Add Database-derived Metadata
    agg_cycle['battery_id'] = row['battery_id']
    agg_cycle['cycle_number'] = row['cycle_number']
    agg_cycle['SoH'] = float((db_capacity / max_capacity) * 100)
    agg_cycle['Re'] = float(row['Re']) if pd.notna(row['Re']) else None
    agg_cycle['Rct'] = float(row['Rct']) if pd.notna(row['Rct']) else None
    agg_cycle['Time_normalized'] = agg_cycle['Time'] / agg_cycle['Time'].max()

    processed_dfs.append(agg_cycle)

# --- Final Dataset Creation ---
total_rows = len(discharge_metadata)
total_kept = len(processed_dfs)
total_skipped = sum(skip_counts.values())

print("--- Cycle processing summary ---")
print(f"Total candidate cycles:   {total_rows}")
print(f"Kept:                     {total_kept} ({100 * total_kept / max(total_rows, 1):.1f}%)")
print(f"Skipped (file not found): {skip_counts['file_not_found']}")
print(f"Skipped (too short):      {skip_counts['too_short']}")
print(f"Skipped (low capacity):   {skip_counts['low_capacity']}")

if total_skipped > 0.5 * total_rows:
    print("WARNING: over half of all cycles were skipped — check cutoff_map / MIN_VALID_CAPACITY_AH.")

if processed_dfs:
    full_dataset = pd.concat(processed_dfs, ignore_index=True)

    # Save the processed data back into a NEW Gold Table for the PINN
    spark_processed = spark.createDataFrame(full_dataset)
    spark_processed.write.format("delta").mode("overwrite").saveAsTable("final_pinn_training_data")

    print("Training database 'final_pinn_training_data' created with target batteries.")
else:
    raise RuntimeError("No cycles survived processing — check the skip-reason counts above.")

In [0]:
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cycle_number ORDER BY SoH DESC) as rn
        FROM final_pinn_training_data
    )
    WHERE rn = 1
    ORDER BY SoH DESC
    LIMIT 10
""").show()

## 4. Physics Definition

**Fix applied:** the linear OCV approximation `U_ocv = 2.7 + 0.015 * SoC` has been replaced with a
4th-order polynomial fit against the pipeline's own `(SoC, Voltage_measured)` low-rate discharge
data. This captures the nonlinear "knees" near 0% and 100% SoC that a straight line misses, which
directly improves the accuracy of the Joule-heating term (`I * (V - U_ocv)`) in the PDE.

If you have access to true open-circuit (near-zero-current, e.g. C/20) reference discharges, refit
`ocv_coeffs` against those instead of the full-rate `Voltage_measured` column for a cleaner OCV
curve — the polynomial fit here is intentionally kept flexible so you can swap the fitting data
without touching the PDE code.

In [0]:
# ---------------------------------------------------------
#  Physics Constants & Parameters
# ---------------------------------------------------------
R_gas = 8.314        # Gas constant (J/mol*K)
Ea_sei = 1.35e5      # Activation energy (J/mol)
H_sei = 2.57e5       # Enthalpy of reaction (J/kg)
rho = 2000.0         # Density (kg/m3)
Cp = 1000.0          # Heat capacity (J/kg*K)
k_cond = 1.0         # Thermal conductivity (W/m*K)
T_ref = 298.15       # Baseline Kelvin (25 degC)

# Define A_sei as a learnable variable (starting at 10^10, i.e. log10(A_sei) = 10.0)
log_A_sei = dde.Variable(10.0)

In [0]:
# ---------------------------------------------------------
#  Data Preparation & Database Querying
# ---------------------------------------------------------
# Load data from the processed SQL database
query = "SELECT * FROM final_pinn_training_data"
df = spark.sql(query).toPandas()

#  Replace any rogue 'Infinite' values with NaN
df = df.replace([np.inf, -np.inf], np.nan)
#  Drop any row that contains missing data in our critical columns
df = df.dropna(subset=[
    'Time', 'Time_normalized', 'Current_measured',
    'Voltage_measured', 'SoC', 'SoH', 'Re', 'Temperature_measured'
])

# Constants for scaling
t_max_seconds = df['Time'].max()

# Strictly monotonic interpolation:
# Since we are training a global model across many batteries,
# we interpolate the average characteristic profiles for Current, Voltage, and SoC.
# This provides the "Representative Discharge" the PINN learns from.

# Sort by normalized time to ensure 1D interpolation works correctly
df_sorted = df.sort_values('Time_normalized')

# Group by normalized time to handle duplicates at specific time steps
df_agg = df_sorted.groupby('Time_normalized').agg({
    'Current_measured': 'mean',
    'Voltage_measured': 'mean',
    'SoC': 'mean'
}).reset_index()

# NOTE: the scipy interp1d objects previously built here have been removed.
# See the "GPU-native interpolation" cell below, which replaces them with
# torch tensors + torch.searchsorted so PDE residual evaluation never has to
# round-trip through the CPU during training.

In [0]:
# ---------------------------------------------------------
#  OCV(SoC) fit  (fix: replaces the linear U_ocv = 2.7 + 0.015*SoC)
# ---------------------------------------------------------
# Fit a 4th-order polynomial to (SoC, Voltage) pairs from the aggregated
# representative discharge curve. This is a pragmatic stand-in for a true
# low-rate OCV curve -- if reference C/20 discharges are available, refit
# against those instead for a cleaner, rate-independent OCV estimate.
_soc_fit = df_agg['SoC'].values / 100.0          # normalize to [0, 1]
_v_fit = df_agg['Voltage_measured'].values

_soc_fit_clipped = np.clip(_soc_fit, 1e-3, 1 - 1e-3)
ocv_coeffs = np.polyfit(_soc_fit_clipped, _v_fit, deg=4)

print(f"Fitted OCV polynomial coefficients (deg=4, highest power first): {ocv_coeffs}")

def ocv_from_soc_torch(soc_tensor):
    """Evaluate the fitted OCV(SoC) polynomial on a torch tensor."""
    soc = torch.clamp(soc_tensor, 1e-3, 1 - 1e-3)
    ocv = torch.zeros_like(soc)
    degree = len(ocv_coeffs) - 1
    for power, coeff in zip(range(degree, -1, -1), ocv_coeffs):
        ocv = ocv + float(coeff) * torch.pow(soc, power)
    return ocv

### GPU-native interpolation (fix: replaces per-call `scipy.interp1d`)

The original `get_real_time_physics` called `.detach().cpu().numpy()` on every invocation, ran
`scipy.interp1d` on the CPU, then moved the result back to the training device. Since this function
is called on every PDE residual evaluation, that CPU/GPU sync happened thousands of times during
training and was very likely the actual training-speed bottleneck — more so than GPU tier choice.

The replacement below precomputes the lookup tables as tensors **once**, on the training device, and
does the interpolation with `torch.searchsorted` so the whole call stays on-device.

In [0]:
# ---------------------------------------------------------
#  GPU-native interpolation tables (fix: replaces scipy interp1d)
# ---------------------------------------------------------
# Precompute sorted breakpoints + values as tensors, on the training device,
# once -- instead of re-running scipy interp1d + a CPU round-trip on every
# single PDE residual call during training.

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

t_pts_t   = torch.tensor(df_agg['Time_normalized'].values, dtype=torch.float32, device=_device)
I_pts_t   = torch.tensor(df_agg['Current_measured'].values, dtype=torch.float32, device=_device)
V_pts_t   = torch.tensor(df_agg['Voltage_measured'].values, dtype=torch.float32, device=_device)
SoC_pts_t = torch.tensor(df_agg['SoC'].values, dtype=torch.float32, device=_device)

def torch_interp(x, xp, fp):
    """
    1D linear interpolation, fully on-device (mirrors np.interp / scipy interp1d
    with linear extrapolation at the edges). x, xp, fp are all torch tensors.
    xp must be sorted ascending (guaranteed since df_agg is sorted by Time_normalized).
    """
    x = x.flatten()
    # Find insertion indices, clamp to valid interior range for interpolation
    idx = torch.searchsorted(xp, x, right=True)
    idx = torch.clamp(idx, 1, len(xp) - 1)

    x0 = xp[idx - 1]
    x1 = xp[idx]
    y0 = fp[idx - 1]
    y1 = fp[idx]

    # Linear interpolation; denom guarded against duplicate breakpoints
    denom = (x1 - x0)
    denom = torch.where(denom.abs() < 1e-12, torch.ones_like(denom), denom)
    slope = (y1 - y0) / denom

    out = y0 + slope * (x - x0)

    # Linear extrapolation beyond the table edges (matches fill_value="extrapolate")
    below = x < xp[0]
    above = x > xp[-1]
    if below.any():
        edge_slope = (fp[1] - fp[0]) / (xp[1] - xp[0] + 1e-12)
        out = torch.where(below, fp[0] + edge_slope * (x - xp[0]), out)
    if above.any():
        edge_slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2] + 1e-12)
        out = torch.where(above, fp[-1] + edge_slope * (x - xp[-1]), out)

    return out.view(-1, 1)


def get_real_time_physics(t_norm):
    """
    Mapping function for the PINN's physics loss.
    Converts normalized time [0,1] into the specific I, V, and OCV
    required to solve the heat equation during training.

    Fully GPU-native: no .detach()/.cpu()/.numpy() round-trip, so this no
    longer forces a CPU<->GPU sync on every PDE residual evaluation.
    """
    t_flat = t_norm.view(-1).to(_device)

    I_val = torch_interp(t_flat, t_pts_t, I_pts_t).to(t_norm.device)
    V_val = torch_interp(t_flat, t_pts_t, V_pts_t).to(t_norm.device)
    SoC_val = torch_interp(t_flat, t_pts_t, SoC_pts_t).to(t_norm.device)

    # OCV: fitted nonlinear curve (from the fix above), evaluated in torch
    U_ocv = ocv_from_soc_torch(SoC_val / 100.0)

    return I_val, V_val, U_ocv

**Optional sanity check** — before relying on this for a full training run, verify the two
interpolation methods agree within floating-point tolerance on the same input points:

```python
from scipy.interpolate import interp1d

_ref_interp_I = interp1d(df_agg['Time_normalized'].values, df_agg['Current_measured'].values, fill_value="extrapolate")
_test_t = np.linspace(0, 1, 200).astype(np.float32)

_ref = _ref_interp_I(_test_t)
_new = torch_interp(torch.tensor(_test_t, device=_device), t_pts_t, I_pts_t).cpu().numpy().flatten()

print("Max abs difference (interp_I vs torch_interp):", np.max(np.abs(_ref - _new)))
```


### 4D Coupled PDE System

Same physics as the original: a heat equation driven by Joule heating + Arrhenius SEI reaction
heat, coupled to a first-order SEI-stability decay equation. Unchanged apart from now calling the
fitted OCV function above.

In [0]:
# ---------------------------------------------------------
# 4D PDE System
# ---------------------------------------------------------

def sei_thermal_pde(x, y):
    T_delta = y[:, 0:1]
    c = y[:, 1:2]

    # 1. Scale and Guard Temperature
    # We use .clamp to ensure the temperature never goes below absolute zero
    # or above a physical limit during the chaotic first steps of Adam.
    T = T_delta * 50.0 + T_ref
    T_safe = torch.clamp(T, min=273.15, max=600.0)

    # 2. Guard Concentration (Health cannot be negative or above 100%)
    c_safe = torch.clamp(c, min=0.0, max=1.0)

    # 3. Gradients
    dT_t = dde.grad.jacobian(y, x, i=0, j=3) / t_max_seconds
    dc_t = dde.grad.jacobian(y, x, i=1, j=3) / t_max_seconds
    dT_xx = dde.grad.hessian(y, x, i=0, j=0)

    # 4. Physics Inputs
    I, V, U_ocv = get_real_time_physics(x[:, 3:4])
    actual_A_sei = torch.pow(10.0, log_A_sei)
    joule_heat = torch.abs(I * (V - U_ocv))

    # 5. Arrhenius Rate with Exponential Guard
    # We clamp the exponent to +/- 50 to prevent torch.exp() from hitting Infinity
    exponent = -Ea_sei / (R_gas * T_safe + 1e-6)
    rate = actual_A_sei * c_safe * torch.exp(torch.clamp(exponent, min=-50, max=50))

    # 6. Thermal Balance (Using a small epsilon 1e-6 to prevent division by zero)
    thermal_mass = (rho * Cp) + 1e-6
    res_thermal = (dT_t * 50.0) - (k_cond * dT_xx * 50.0 + joule_heat + H_sei * rate) / thermal_mass

    res_sei = dc_t + rate

    return [res_thermal, res_sei]

## 5. Geometry & Boundary/Initial Conditions

In [0]:
# ---------------------------------------------------------
# Geometry & Robust Initial Conditions
# ---------------------------------------------------------

# 1. Define Geometry
param_geom = dde.geometry.Hypercube(xmin=[0, 0.5, 0.0], xmax=[1, 1.2, 12.0])
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(param_geom, timedomain)

# 2. Fuzzy ICs
def on_initial_condition(x, on_initial):
    # Using on_initial (the built-in flag) OR a small temporal window
    return on_initial or (x[3] <= 0.01)

ic_T = dde.icbc.IC(geomtime, lambda x: 0, on_initial_condition, component=0)
ic_c = dde.icbc.IC(geomtime, lambda x: 1.0, on_initial_condition, component=1)

# --- Observation points for the data-fitting boundary condition ---
obs_points = np.column_stack((
    np.ones(len(df)),             # Space (x=1)
    df['SoH'].values / 100.0,     # SoH
    df['Re'].values * 10.0,       # Re
    df['Time_normalized'].values  # Time (t)
))

obs_values = ((df['Temperature_measured'].values - T_ref) / 50.0).reshape(-1, 1)

# Synchronize Data Types
obs_points = obs_points.astype(np.float32)
obs_values = obs_values.astype(np.float32)

observe_T = dde.icbc.PointSetBC(obs_points, obs_values, component=0)

## 6. Model Architecture & Training

**Fix applied:** at the end of training, `log_A_sei` (the calibrated Arrhenius parameter) is saved
*inside* the same checkpoint dict as the network weights, instead of only the weights being
persisted. This is what makes correct restoration possible in Section 7.

In [0]:
import mlflow
import torch
import deepxde as dde

# ---------------------------------------------------------
# Model Architecture & Training
# ---------------------------------------------------------
# Metrics for Battery Digital Twin
def mae_temp_metric(y_true, y_pred):
    # If DeepXDE is testing on PDE points (no NASA labels), y_true is None
    if y_true is None:
        return torch.tensor(0.0)

    # y_pred has 2 columns: [T_delta, c]
    # y_true has 1 column: [Temperature_measured]
    # We slice y_pred to only compare the Temperature component
    return torch.mean(torch.abs(y_true - y_pred[:, 0:1]))

def l2_temp_error(y_true, y_pred):
    if y_true is None:
        return torch.tensor(0.0)

    # Compare only the temperature components
    return torch.norm(y_true - y_pred[:, 0:1]) / (torch.norm(y_true) + 1e-7)

# Data Object Configuration
# num_test=2000 provides the 'y_true' values needed for the metrics to function
data = dde.data.TimePDE(
    geomtime,
    sei_thermal_pde,
    [ic_T, ic_c, observe_T],
    num_domain=5000,
    num_boundary=1000,    # Doubled this to ensure t=0 is captured
    anchors=obs_points,
    num_test=2000
)

# Network Architecture
# 4 Inputs: [Space, SoH, Re, Time] -> 4 Hidden Layers -> 2 Outputs: [T_delta, c]
# Use the float32 casting for the network weights
net = dde.nn.FNN([4] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float()  # Force weights to float32
model = dde.model.Model(data, net)

# Custom MLflow Logging Callback
class MLflowLoggingCallback(dde.callbacks.Callback):
    def on_epoch_end(self):
        # Log every 1000 steps to avoid cluttering Databricks logs
        if self.model.train_state.step % 1000 == 0:
            # Log the learned physical parameter A_sei
            current_A = 10 ** log_A_sei.detach().item()
            mlflow.log_metric("A_sei_value", current_A, step=self.model.train_state.step)

            # Log all 5 loss components (PDE 1, PDE 2, IC 1, IC 2, Data)
            for i, loss in enumerate(self.model.train_state.loss_train):
                mlflow.log_metric(f"loss_comp_{i}", float(loss), step=self.model.train_state.step)

            # Log the metrics (MAE and L2)
            if self.model.train_state.metrics_test is not None:
                mlflow.log_metric("MAE_Kelvin_Scaled", float(self.model.train_state.metrics_test[0]), step=self.model.train_state.step)
                mlflow.log_metric("L2_Relative_Error", float(self.model.train_state.metrics_test[1]), step=self.model.train_state.step)


# Execution within MLflow Context
with mlflow.start_run(run_name="Global_Battery_Digital_Twin"):

    print("Starting Adam Optimization...")
    # Stage 1: Adam Optimizer
    model.compile(
        "adam",
        lr=1e-4,  # Lower learning rate to prevent early 'jump' to NaN
        loss_weights=[1, 1, 10, 10, 100],
        external_trainable_variables=log_A_sei,
        metrics=[mae_temp_metric, l2_temp_error]
    )

    # Train with Adam for 15,000 iterations to stabilize the weights
    losshistory_adam, train_state_adam = model.train(iterations=15000, callbacks=[MLflowLoggingCallback()])

    # Stage 2: L-BFGS Fine-Tuning
    print("Starting L-BFGS Refinement...")
    model.compile("L-BFGS", external_trainable_variables=log_A_sei)

    # Increase the tolerance so L-BFGS doesn't give up early
    dde.optimizers.config.set_LBFGS_options(maxiter=2000, ftol=1e-9)

    # Train with L-BFGS to fine-tune the stiff PDE gradients
    losshistory_lbfgs, train_state_lbfgs = model.train()

    # Final Artifact Logging
    final_A = 10 ** log_A_sei.detach().item()
    mlflow.log_param("Final_Calibrated_A_sei", f"{final_A:.2e}")

    # --- Fix: persist log_A_sei alongside the network weights ---
    # Previously only net.state_dict() was saved, so restoring the model for
    # inference reset log_A_sei to its untrained default (10.0) rather than
    # the value actually calibrated during training.
    torch.save({
        "model_state_dict": net.state_dict(),
        "log_A_sei": log_A_sei.detach().item(),
        "ocv_coeffs": ocv_coeffs.tolist(),
    }, "battery_pinn_model.pt")
    mlflow.log_artifact("battery_pinn_model.pt")

print(f"Training successfully completed. Frequency factor calibrated to: {final_A:.2e}")
model.save("global_pinn_model")

## 7. Digital Twin Restoration & Inference

**Fixes applied:**
- The real `sei_thermal_pde` (rather than a `dummy_pde` placeholder) is reused to reconstruct the
  network architecture, so the PDE machinery is available if you want to check residuals later.
- `log_A_sei` and the fitted `ocv_coeffs` are restored from the checkpoint, not left at their
  untrained defaults.
- An explicit note clarifies what `model.predict()` does and does not do at inference time.

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow
from mlflow.tracking import MlflowClient

# 1. SET BACKEND BEFORE IMPORTING DEEPXDE
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde

# Constants for Denormalization
T_ref = 298.15
T_scale = 50.0

# 2. RECONSTRUCT ARCHITECTURE using the REAL physics function.
#
# NOTE: dde.data.TimePDE needs *a* PDE function to construct the Data object,
# but at inference time model.predict() only performs a forward pass through
# the trained network -- it does NOT re-solve or enforce the PDE residual.
# Physics constraints were baked into the weights during training only.
# We reuse sei_thermal_pde here (instead of a disconnected dummy function)
# purely so the same Data/Model objects can optionally be used later to
# evaluate PDE residuals on new predictions, e.g. as a sanity check.
geom = dde.geometry.Interval(0, 1)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)
data = dde.data.TimePDE(geomtime, sei_thermal_pde, [], num_domain=100)
net = dde.nn.FNN([4] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float()

model = dde.model.Model(data, net)

# 3. DOWNLOAD & RESTORE
client = MlflowClient()
run_id = "49ce940bc4e64a42a56311add3e7fc13"
artifact_path = "battery_pinn_model.pt"

local_path = client.download_artifacts(run_id, artifact_path)
checkpoint = torch.load(local_path, map_location=torch.device('cpu'))

# Robust load for DeepXDE format vs standard PyTorch
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.net.load_state_dict(checkpoint["model_state_dict"])

    # --- Fix: restore log_A_sei from checkpoint instead of using the
    # untrained module-level default (10.0) ---
    if "log_A_sei" in checkpoint:
        log_A_sei = dde.Variable(checkpoint["log_A_sei"])
        print(f"Restored A_sei = {10 ** checkpoint['log_A_sei']:.3e}")
    else:
        print("WARNING: no log_A_sei found in checkpoint -- using untrained default (10.0).")

    # --- Fix: restore the fitted OCV polynomial coefficients too ---
    if "ocv_coeffs" in checkpoint:
        ocv_coeffs = np.array(checkpoint["ocv_coeffs"])
        print("Restored fitted OCV polynomial coefficients.")
    else:
        print("WARNING: no ocv_coeffs found in checkpoint -- OCV function may be undefined.")
else:
    model.net.load_state_dict(checkpoint)
    print("WARNING: legacy checkpoint format -- log_A_sei and ocv_coeffs were not restored.")

model.compile(
    "adam",
    lr=1e-3,
    loss_weights=[1, 1, 1, 1, 100],
    external_trainable_variables=log_A_sei,
    metrics=[mae_temp_metric, l2_temp_error]
)

print("Model Restored. Architecture: 4 Hidden Layers. Backend: PyTorch.")

In [0]:
# 4. DIGITAL TWIN SIMULATION
test_t = np.linspace(0, 1, 100)
s_soh, s_re = 0.8, 1.5  # Simulate an 80% health battery

# Predict Internal (x=0) vs Surface (x=1)
core_in = np.column_stack((
    np.zeros(100),           # Space (x=0)
    np.full(100, s_soh),     # SoH
    np.full(100, s_re),      # Re
    test_t                   # Time
)).astype(np.float32)

core_pred = model.predict(core_in)
print("Sample digital-twin prediction (first 5 rows, [T_delta, c]):")
print(core_pred[:5])

### Abuse Test — "Point of No Return" Simulation

Sweeps the input temperature slot from 25 to 150 C to find where the predicted SEI stability `c`
collapses, as a proxy for thermal runaway risk. Unchanged from the original except for the comment
clarifying that this is a pure forward-pass prediction against a trained network, not a live PDE
solve.

In [0]:
# --- Simulation of Ignition Threshold ---
# We force the temperature to rise to see when 'c' collapses.
# Reminder: this is model.predict() -- a forward pass through the trained
# network -- not a re-solved PDE. The physics is encoded in the learned weights.
oven_t_c = np.linspace(25, 150, 100)  # Simulating an oven test 25C to 150C
oven_t_norm = (oven_t_c + 273.15 - T_ref) / 50.0

# Input: [x=0, SoH=0.8, Re=0.5, t_norm]
abuse_test = np.column_stack((
    np.zeros(100),               # Core
    np.full(100, 0.8),           # 80% SoH
    np.full(100, 0.5),           # Standard Re
    np.linspace(0, 1, 100)       # Time steps
)).astype(np.float32)

res = model.predict(abuse_test)
core_temp = res[:, 0] * 50 + (T_ref - 273.15)
sei_health = res[:, 1]

# Logic based on the "Point of No Return" graph
if core_temp.max() > 80.0 or sei_health.min() < 0.985:
    status = "DANGER: THERMAL RUNAWAY PREDICTED"
elif core_temp.max() > 60.0:
    status = "WARNING: ACCELERATED AGING DETECTED"
else:
    status = "SAFE"

print(f"Status: {status}")

# Visualisation
plt.figure(figsize=(10, 5))
plt.plot(core_temp, sei_health, 'g-', label='SEI Health')
plt.axvline(x=80, color='r', linestyle='--', label='Critical Threshold (80C)')
plt.xlabel('Internal Temperature (degC)')
plt.ylabel('Chemical Stability (c)')
plt.title('Digital Twin: Predicting the Point of No Return')
plt.axhline(y=0.985, color='orange', linestyle=':', label='Chemical Failure Limit')
plt.legend()
plt.show()